# Data Cleaning Deep Dive

## Why Data Cleaning Matters

Data cleaning (also called "data normalization" or "data wrangling") is where data scientists spend **60-80% of their time**. Raw data is messy — especially when it comes from different sources:

- A receipt says: `"Greenwise Hmstyle Meatbal Ft"`
- Your pantry scan says: `"meatball"`
- A recipe says: `"meatballs"`

These are all the **same food** but described three different ways! Without cleaning, a computer can't tell they match. This notebook walks through the cleaning process step by step, showing how each transformation improves match rates.

## Step 1: Loading the Raw Data

Let's start by looking at the messy, unprocessed data — exactly as it came from receipts, pantry scans, and recipes.

In [ ]:
import sys
import json
import re
from pathlib import Path

# Add the project root to Python's module search path
project_root = str(Path.cwd().parent) if Path.cwd().name == "notebooks" else str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
from src.database import get_engine

engine = get_engine()

# Load raw receipt data (messy names from store POS systems)
receipts_df = pd.read_sql("SELECT item_name, normalized_name, category FROM receipt", engine)

# Load raw pantry data (cleaner names from AI extraction)
pantry_df = pd.read_sql("SELECT item_name, category FROM pantryitem", engine)

# Load recipe ingredients
recipes_df = pd.read_sql("SELECT name, ingredients FROM recipe", engine)

# Extract ingredient names from JSON
recipes_df["parsed_ingredients"] = recipes_df["ingredients"].apply(
    lambda x: json.loads(x) if isinstance(x, str) else x
)

all_recipe_ingredients = set()
for _, recipe in recipes_df.iterrows():
    for ing in recipe["parsed_ingredients"]:
        if isinstance(ing, dict) and "name" in ing:
            all_recipe_ingredients.add(ing["name"].lower().strip())

print(f"Receipt items: {len(receipts_df)} rows, {receipts_df['item_name'].nunique()} unique names")
print(f"Pantry items: {len(pantry_df)} rows, {pantry_df['item_name'].nunique()} unique names")
print(f"Recipe ingredients: {len(all_recipe_ingredients)} unique ingredients")

## Step 2: The Messy State

Let's see what we're working with. Notice how different the same food looks across sources:

In [ ]:
# Show sample receipt names — these are the messiest
print("=== RAW RECEIPT NAMES ===")
print("(These come from store POS systems — abbreviated, coded, branded)\n")
for name in receipts_df["item_name"].unique()[:15]:
    print(f"  '{name}'")

print(f"\n=== RAW PANTRY NAMES ===")
print("(These are cleaner — AI-extracted from photos)\n")
for name in pantry_df["item_name"].unique()[:15]:
    print(f"  '{name}'")

print(f"\n=== RECIPE INGREDIENT NAMES ===")
print("(These are the target format — clean, standard names)\n")
for name in sorted(all_recipe_ingredients)[:15]:
    print(f"  '{name}'")

In [ ]:
# HOW DO WE MEASURE SUCCESS?
# Match rate = what percentage of recipe ingredients can we find in our inventory?
# Before any cleaning, we just do exact string matching.

def calculate_match_rate(inventory_names: set, recipe_ingredients: set) -> float:
    """Calculate what percentage of recipe ingredients match inventory names."""
    if not recipe_ingredients:
        return 0.0
    matches = inventory_names & recipe_ingredients  # Set intersection: items in BOTH
    return len(matches) / len(recipe_ingredients) * 100

# Combine all raw inventory names (receipts + pantry)
raw_receipt_names = set(receipts_df["item_name"].str.lower().str.strip())
raw_pantry_names = set(pantry_df["item_name"].str.lower().str.strip())
raw_inventory = raw_receipt_names | raw_pantry_names  # Union: items in EITHER

# Use normalized_name from receipts if available (AI already cleaned these)
if "normalized_name" in receipts_df.columns:
    norm_receipt_names = set(receipts_df["normalized_name"].dropna().str.lower().str.strip())
    raw_inventory = raw_inventory | norm_receipt_names

initial_rate = calculate_match_rate(raw_inventory, all_recipe_ingredients)
print(f"Before cleaning: {initial_rate:.0f}% of recipe ingredients found in inventory")
print(f"  ({int(initial_rate * len(all_recipe_ingredients) / 100)} out of {len(all_recipe_ingredients)} ingredients)")

# Which ones matched?
initial_matches = raw_inventory & all_recipe_ingredients
if initial_matches:
    print(f"\n  Matched: {sorted(initial_matches)}")

# Which ones didn't?
initial_missing = all_recipe_ingredients - raw_inventory
if initial_missing:
    print(f"\n  NOT matched ({len(initial_missing)}): {sorted(initial_missing)[:20]}")

## Step 3: Cleaning Step by Step

Now let's apply cleaning transformations one at a time and watch the match rate improve. Each step removes a specific type of noise from the data.

In [ ]:
# CLEANING STEP 1: Lowercase everything
# Why? "Butter" and "butter" are the same food but won't match as strings.
# This is the simplest and most impactful cleaning step.

def clean_lowercase(name: str) -> str:
    """Convert to lowercase."""
    return name.lower().strip()

# Apply to all inventory names
step1_names = set()
for name in raw_receipt_names | raw_pantry_names:
    step1_names.add(clean_lowercase(name))

# Also add normalized receipt names
if "normalized_name" in receipts_df.columns:
    for name in receipts_df["normalized_name"].dropna():
        step1_names.add(clean_lowercase(name))

step1_rate = calculate_match_rate(step1_names, all_recipe_ingredients)
print(f"After lowercase: {step1_rate:.0f}% match rate")
print(f"  Improvement: +{step1_rate - initial_rate:.0f} percentage points")

In [ ]:
# CLEANING STEP 2: Remove brand names, qualifiers, and marketing terms
# Why? "Publix Organic Free Range Eggs" should just be "eggs"

# Load the qualifiers list from the config file
# We use the project_root variable set in the import cell
import os
config_path = os.path.join(project_root, "config", "normalization_mappings.json")
with open(config_path) as f:
    config = json.load(f)

# The config stores qualifiers as a dict of categories -> lists
# We need to flatten all the sub-lists into one list of terms to strip
qualifiers_config = config.get("qualifiers_to_strip", {})
qualifiers = []
for key, val in qualifiers_config.items():
    if isinstance(val, list):
        qualifiers.extend(val)

packaging_config = config.get("packaging_terms_to_strip", {})
packaging = []
for key, val in packaging_config.items():
    if isinstance(val, list):
        packaging.extend(val)

print(f"Qualifiers to strip ({len(qualifiers)} terms):")
print(f"  {qualifiers[:10]}...")
print(f"\nPackaging terms to strip ({len(packaging)} terms):")
print(f"  {packaging[:10]}...")

def clean_remove_qualifiers(name: str) -> str:
    """Remove brand names, qualifiers, and marketing terms."""
    name = clean_lowercase(name)
    # Remove each qualifier word from the name
    for q in qualifiers + packaging:
        # Use word boundary regex so "organic" doesn't remove "organ" from "oregano"
        name = re.sub(r'\b' + re.escape(q.lower()) + r'\b', '', name)
    # Clean up extra whitespace left behind
    name = re.sub(r'\s+', ' ', name).strip()
    return name

# Apply to all names
step2_names = set()
for name in raw_receipt_names | raw_pantry_names:
    step2_names.add(clean_remove_qualifiers(name))
if "normalized_name" in receipts_df.columns:
    for name in receipts_df["normalized_name"].dropna():
        step2_names.add(clean_remove_qualifiers(name))

step2_rate = calculate_match_rate(step2_names, all_recipe_ingredients)
print(f"\nAfter removing qualifiers: {step2_rate:.0f}% match rate")
print(f"  Improvement: +{step2_rate - step1_rate:.0f} percentage points from this step")

In [ ]:
# CLEANING STEP 3: Remove sizes, weights, and count patterns
# Why? "16oz butter" and "1lb butter" should both just be "butter"

size_patterns_config = config.get("size_patterns_to_strip", {})
size_patterns = []
for key, val in size_patterns_config.items():
    if isinstance(val, list):
        size_patterns.extend(val)

print(f"Size patterns to strip ({len(size_patterns)} patterns):")
for p in size_patterns[:5]:
    print(f"  {p}")

def clean_remove_sizes(name: str) -> str:
    """Remove size, weight, and count patterns."""
    name = clean_remove_qualifiers(name)
    for pattern in size_patterns:
        try:
            name = re.sub(pattern, '', name, flags=re.IGNORECASE)
        except re.error:
            pass  # Skip invalid regex patterns
    name = re.sub(r'\s+', ' ', name).strip()
    return name

# Apply
step3_names = set()
for name in raw_receipt_names | raw_pantry_names:
    step3_names.add(clean_remove_sizes(name))
if "normalized_name" in receipts_df.columns:
    for name in receipts_df["normalized_name"].dropna():
        step3_names.add(clean_remove_sizes(name))

step3_rate = calculate_match_rate(step3_names, all_recipe_ingredients)
print(f"\nAfter removing sizes: {step3_rate:.0f}% match rate")
print(f"  Improvement: +{step3_rate - step2_rate:.0f} percentage points from this step")

In [ ]:
# CLEANING STEP 4: Expand receipt abbreviations
# Why? Receipts use codes like "chkn" for "chicken", "bnls" for "boneless"

abbreviations = {k: v for k, v in config.get("abbreviations", {}).items()
                 if k != "_description" and isinstance(v, str)}

print(f"Abbreviation expansions ({len(abbreviations)} mappings):")
for abbr, full in list(abbreviations.items())[:8]:
    print(f"  '{abbr}' → '{full}'")

def clean_expand_abbreviations(name: str) -> str:
    """Expand receipt abbreviations to full words."""
    name = clean_remove_sizes(name)
    for abbr, full in abbreviations.items():
        name = re.sub(r'\b' + re.escape(abbr.lower()) + r'\b', full.lower(), name)
    name = re.sub(r'\s+', ' ', name).strip()
    return name

# Apply
step4_names = set()
for name in raw_receipt_names | raw_pantry_names:
    step4_names.add(clean_expand_abbreviations(name))
if "normalized_name" in receipts_df.columns:
    for name in receipts_df["normalized_name"].dropna():
        step4_names.add(clean_expand_abbreviations(name))

step4_rate = calculate_match_rate(step4_names, all_recipe_ingredients)
print(f"\nAfter expanding abbreviations: {step4_rate:.0f}% match rate")
print(f"  Improvement: +{step4_rate - step3_rate:.0f} percentage points from this step")

In [ ]:
# CLEANING STEP 5: Resolve aliases to canonical names
# Why? "chkn breast" and "boneless chicken" should both become "chicken breast"

aliases = {k: v for k, v in config.get("name_aliases", {}).items()
           if k != "_description" and isinstance(v, list)}

print(f"Alias groups ({len(aliases)} canonical names):")
for canonical, alias_list in list(aliases.items())[:5]:
    print(f"  '{canonical}' ← {alias_list[:3]}...")

# Build a reverse lookup: alias → canonical name
alias_to_canonical = {}
for canonical, alias_list in aliases.items():
    for alias in alias_list:
        alias_to_canonical[alias.lower().strip()] = canonical.lower().strip()
    # The canonical name maps to itself
    alias_to_canonical[canonical.lower().strip()] = canonical.lower().strip()

def clean_resolve_aliases(name: str) -> str:
    """Resolve known aliases to their canonical name."""
    name = clean_expand_abbreviations(name)
    # Check if the cleaned name matches any known alias
    if name in alias_to_canonical:
        return alias_to_canonical[name]
    return name

# Apply
step5_names = set()
for name in raw_receipt_names | raw_pantry_names:
    step5_names.add(clean_resolve_aliases(name))
if "normalized_name" in receipts_df.columns:
    for name in receipts_df["normalized_name"].dropna():
        step5_names.add(clean_resolve_aliases(name))

step5_rate = calculate_match_rate(step5_names, all_recipe_ingredients)
print(f"\nAfter alias resolution: {step5_rate:.0f}% match rate")
print(f"  Improvement: +{step5_rate - step4_rate:.0f} percentage points from this step")

## Step 4: Match Rate Summary

Let's see the cumulative improvement from each cleaning step:

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

# Collect all the rates
steps = ["Raw", "Lowercase", "Remove\nQualifiers", "Remove\nSizes",
         "Expand\nAbbreviations", "Resolve\nAliases"]
rates = [initial_rate, step1_rate, step2_rate, step3_rate, step4_rate, step5_rate]

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.bar(steps, rates, color=["#e74c3c"] + ["#3498db"] * 4 + ["#27ae60"],
              edgecolor="white", width=0.6)

# Add percentage labels on bars
for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f"{rate:.0f}%", ha="center", fontweight="bold", fontsize=12)

ax.set_ylabel("Match Rate (%)")
ax.set_title("How Each Cleaning Step Improves Match Rates")
ax.set_ylim(0, 100)

# Add a horizontal line at 100% for reference
ax.axhline(y=100, color="gray", linestyle="--", alpha=0.5, label="100% (perfect)")
ax.legend()

plt.tight_layout()
plt.show()

# Print the summary
print("\nCumulative improvement:")
for step_name, rate in zip(steps, rates):
    improvement = rate - rates[0]
    bar = "█" * int(rate / 2)
    print(f"  {step_name.replace(chr(10), ' '):20s} {rate:5.1f}%  {bar}  (+{improvement:.0f}pp)")

## Step 5: Using the Real Normalizer

The project includes a proper normalizer at `src/normalization/food_names.py` that combines all these steps (and more) into a single function. Let's see how it compares:

In [ ]:
from src.normalization.food_names import normalize_food_name

# Apply the real normalizer to all inventory names
real_normalized = set()

for name in receipts_df["item_name"].unique():
    real_normalized.add(normalize_food_name(name))

if "normalized_name" in receipts_df.columns:
    for name in receipts_df["normalized_name"].dropna().unique():
        real_normalized.add(normalize_food_name(name))

for name in pantry_df["item_name"].unique():
    real_normalized.add(normalize_food_name(name))

real_rate = calculate_match_rate(real_normalized, all_recipe_ingredients)

print(f"Real normalizer match rate: {real_rate:.0f}%")
print(f"\nBefore/After comparison:")
print(f"  {'Item Name':<35s} → {'Normalized'}")
print(f"  {'-'*35}   {'-'*25}")

for name in list(receipts_df["item_name"].unique())[:10]:
    normalized = normalize_food_name(name)
    print(f"  {name:<35s} → {normalized}")

## Step 6: Investigating What's Still Not Matching

Even after cleaning, some items won't match. Understanding WHY helps you add new rules to improve coverage.

In [ ]:
# What recipe ingredients still can't be found in cleaned inventory?
still_missing = all_recipe_ingredients - real_normalized

print(f"Still missing ({len(still_missing)} ingredients):")
for ing in sorted(still_missing):
    # Is there something CLOSE in inventory?
    close_matches = [inv for inv in real_normalized 
                     if ing in inv or inv in ing]
    if close_matches:
        print(f"  '{ing}' — close inventory match: {close_matches}")
    else:
        print(f"  '{ing}' — no close match in inventory")

print(f"\nTo improve match rates:")
print(f"  1. Add new aliases to config/normalization_mappings.json")
print(f"  2. Buy the missing ingredients!")
print(f"  3. Add more recipes that use what you already have")

## Exercises

1. **Add a cleaning rule for a pattern specific to your data.** Look at the "still missing" list above. Can you write a regex or alias that would match one of those items?

2. **Find two items that SHOULD match but don't.** Look at your inventory names and recipe ingredient names. What cleaning rule would bridge the gap? Add it to `config/normalization_mappings.json` and re-run this notebook.

3. **What's the match rate if you only use category-level matching instead of exact name?** Instead of matching "cheddar cheese" exactly, count it as a match if ANY "dairy" item exists in inventory. How much does this improve coverage?

*Hint for exercise 3*: Load the active inventory and recipe ingredient match tables. For each ingredient, check if its category has any items in inventory, regardless of exact name.